### Installation Step

In [16]:
import os
from dotenv import load_dotenv
load_dotenv()

# Load environment variables from .env file
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')


In [17]:
from litellm import completion
import litellm

In [27]:
# Call OpenAI fro Groq (super fast inference)
response_openai = completion(
    model="groq/openai/gpt-oss-20b",
    messages=[{"role": "user", "content": "Explain RAG in one sentence."}]
)
print("OpenAI:      ", response_openai.choices[0].message.content)

OpenAI:       RAG (Retrieval‑Augmented Generation) is a technique that blends a retrieval system, which fetches relevant external documents, with a generative language model, allowing the model to incorporate that retrieved information into its responses.


### Unified function for multiple LLM providers

In [20]:
# One function call to invoke multiple LLM providers

from litellm import completion

prompt = "Explain RAG in one sentence."

# Just a list of model strings — that's the only configuration
providers = [
    ("Groq",       "groq/llama-3.3-70b-versatile"),
    ("OpenAI",     "gpt-4o-mini"),
    ("🟡 Gemini",     "gemini/gemini-1.5-flash"),
]

# ONE loop. ONE function call. Multiple providers.
for label, model in providers:
    try:
        r = completion(model=model, messages=[{"role": "user", "content": prompt}])
        print(f"{label:<15}: {r.choices[0].message.content[:80]}")
    except Exception as e:
        print(f"{label:<15}: ❌ {type(e).__name__}")

Groq           : RAG (Retrieve, Augment, Generate) is an artificial intelligence framework that c

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

OpenAI         : ❌ InternalServerError

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers

🟡 Gemini       : ❌ APIConnectionError


### Fallback Mechanism

In [21]:
from litellm import completion

# Define a fallback chain: try Gemini first, then Groq
response = completion(
    model="gemini/gemini-1.5-flash",
    messages=[{"role": "user", "content": "What is an LLM Gateway?"}],
    fallbacks=[
        "groq/llama-3.3-70b-versatile"
    ]
)

print("Response:", response.choices[0].message.content[:200], "...")
print("\nWhich model actually answered?", response.model)

22:51:38 - LiteLLM:ERROR: fallback_utils.py:68 - Fallback attempt failed for model gemini/gemini-1.5-flash: litellm.APIConnectionError: Missing Gemini API key. Set the GEMINI_API_KEY or GOOGLE_API_KEY environment variable.
Traceback (most recent call last):
  File "c:\Users\T0311J8\Downloads\LLM Gateway\.venv\Lib\site-packages\litellm\main.py", line 637, in acompletion
    response = await init_response
               ^^^^^^^^^^^^^^^^^^^
  File "c:\Users\T0311J8\Downloads\LLM Gateway\.venv\Lib\site-packages\litellm\llms\vertex_ai\gemini\vertex_and_google_ai_studio_gemini.py", line 2955, in async_completion
    auth_header, api_base = self._get_token_and_url(
                            ~~~~~~~~~~~~~~~~~~~~~~~^
        model=model,
        ^^^^^^^^^^^^
    ...<9 lines>...
        use_psc_endpoint_format=use_psc_endpoint_format,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "c:\Users\T0311J8\Downloads\LLM Gateway\.venv\Lib\site-packages\litellm\llms\vertex_a


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



c:\Users\T0311J8\Downloads\LLM Gateway\.venv\Lib\site-packages\litellm\litellm_core_utils\logging_worker.py:75: RuntimeWarning: coroutine 'Logging.async_success_handler' was never awaited
  self._queue = None
Task was destroyed but it is pending!
task: <Task pending name='Task-95' coro=<LoggingWorker._worker_loop() running at c:\Users\T0311J8\Downloads\LLM Gateway\.venv\Lib\site-packages\litellm\litellm_core_utils\logging_worker.py:110>>


Response: An LLM (Large Language Model) Gateway is an interface or a platform that allows users to interact with large language models, such as those used in artificial intelligence (AI) and natural language pr ...

Which model actually answered? llama-3.3-70b-versatile


c:\Users\T0311J8\Downloads\LLM Gateway\.venv\Lib\site-packages\litellm\litellm_core_utils\logging_worker.py:77: RuntimeWarning: coroutine 'LoggingWorker._worker_loop' was never awaited
  self._worker_task = None


### Caching

In [31]:
import litellm

# 🧹 Reset any callbacks/strategies left over from earlier cells
litellm.callbacks = []
litellm.success_callback = []
litellm.failure_callback = []
litellm._async_success_callback = []
litellm._async_failure_callback = []

# Also clear any router-strategy state
litellm.cache = None

print("LiteLLM state reset — ready for clean caching demo")

LiteLLM state reset — ready for clean caching demo


In [33]:
import litellm
import time
from litellm import completion
from litellm.caching import Cache

# Enable in-memory caching (you can also use Redis in production)
litellm.cache = Cache(type="local")

prompt = "What is AI? Answer in one line."

# First call — actually hits OpenAI
start = time.time()
r1 = completion(
    model="groq/llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}],
    caching=True
)
t1 = time.time() - start
print(f"First call (API):   {t1:.2f}s — {r1.choices[0].message.content}")

# Second call — served from cache, near-instant
start = time.time()
r2 = completion(
    model="groq/llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}],
    caching=True
)
t2 = time.time() - start
print(f"Second call (cache): {t2:.4f}s — {r2.choices[0].message.content}")

First call (API):   0.59s — AI, or Artificial Intelligence, refers to the development of computer systems that can perform tasks that typically require human intelligence, such as learning, reasoning, and problem-solving.
Second call (cache): 0.0041s — AI, or Artificial Intelligence, refers to the development of computer systems that can perform tasks that typically require human intelligence, such as learning, reasoning, and problem-solving.

Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers

